# Extract Four-Tuples from Two CSV Files

This notebook reads two CSV files (`faicomb.csv` and `ffaicomb.csv`) and extracts four fields for each relationship:
- `thread_user_pk`
- `thread_follower_pk`
- `thread_username` (empty if not available)
- `thread_follower_username`

For the first file, it uses `user_pk` and `follower_pk`.  
For the second file, it prefers `user_threads_userpk` for `thread_user_pk` if present, falling back to `user_pk`.  
Resulting data are concatenated and saved as `extracted_four_tuples.csv`.


In [53]:
import pandas as pd
df0 = pd.read_csv('data.csv')
df0 = df0[['user_pk',"username","thread_user_pk"]]

In [98]:
df2 = pd.read_csv('ffaicomb.csv')
df2 = df2.rename(columns={'user_threads_userpk': 'thread_user_pk'})

# 2) Fai il merge con df0 per prendere username e il thread_user_pk di backup
df2 = pd.merge(
    df2,
    pd.merge(pd.read_csv('faicomb.csv'), df0, on='user_pk', how='left')[["follower_pk","follower_username"]],
    left_on='user_pk',   
    right_on='follower_pk',
    how='left',
    suffixes=('', '_df0')
)

# # 3) Riempi i NaN di thread_user_pk con quelli presi da df0
df2['thread_user_pk'] = df2['thread_user_pk'].fillna(df2['follower_pk_df0'])

# # 4) Pulisci le colonne ausiliarie
df2 = df2.drop(columns=['follower_pk_df0'])
df2 = df2[df2['thread_user_pk'].notna()]
df2

,user_pk,follower_pk,follower_username,follower_count,thread_user_pk,follower_username_df0
0,1451797369,5.259982e+10,rejectgirl_,0.0,1.451797e+09,stacidblack
1,1451797369,5.259982e+10,rejectgirl_,0.0,1.451797e+09,stacidblack
2,1451797369,2.064215e+08,awallace253,125.0,1.451797e+09,stacidblack
3,1451797369,2.064215e+08,awallace253,125.0,1.451797e+09,stacidblack
4,1451797369,3.024873e+08,rosario.zavala.mx,76.0,1.451797e+09,stacidblack
...,...,...,...,...,...,...
757874,3293736049,6.326939e+10,lottie._.aashu,33.0,6.682342e+10,elxargus_xyz
757875,3293736049,6.307821e+10,daniza1996,88.0,6.682342e+10,elxargus_xyz
757876,3293736049,6.307821e+10,daniza1996,88.0,6.682342e+10,elxargus_xyz
757877,6580358,6.345378e+10,NaN,NaN,6.345378e+10,paulasparadise


In [ ]:
df2 = df2.rename(columns={'follower_username_df0': 'thread_username'})
df2["follower_pk"] = df2["follower_pk"].apply(lambda x: int(x) if pd.notna(x) else x)
df2["thread_user_pk"] = df2["thread_user_pk"].apply(lambda x: int(x) if pd.notna(x) else x)


In [116]:
df2

,user_pk,follower_pk,follower_username,follower_count,thread_user_pk,thread_username
0,1451797369,5.259982e+10,rejectgirl_,0.0,1451797369,stacidblack
1,1451797369,5.259982e+10,rejectgirl_,0.0,1451797369,stacidblack
2,1451797369,2.064215e+08,awallace253,125.0,1451797369,stacidblack
3,1451797369,2.064215e+08,awallace253,125.0,1451797369,stacidblack
4,1451797369,3.024873e+08,rosario.zavala.mx,76.0,1451797369,stacidblack
...,...,...,...,...,...,...
722397,3293736049,6.326939e+10,lottie._.aashu,33.0,66823423030,elxargus_xyz
722398,3293736049,6.307821e+10,daniza1996,88.0,66823423030,elxargus_xyz
722399,3293736049,6.307821e+10,daniza1996,88.0,66823423030,elxargus_xyz
722400,6580358,6.345378e+10,NaN,NaN,63453776004,paulasparadise


In [123]:


# Read input files
df1 = pd.read_csv('faicomb.csv')
df2 = pd.read_csv('ffaicomb.csv')

# Map columns for file1
df1_mapped = pd.DataFrame({
    'thread_user_pk': df1.get('thread_user_pk', df1['user_pk']),
    'thread_follower_pk': df1.get('thread_follower_pk', df1['follower_pk']),
    'thread_username': df1.get('thread_username', df1['username']),
    'thread_follower_username': df1.get('thread_follower_username', df1['follower_username'])
})

# Map columns for file2
df2_mapped = pd.DataFrame({
    'thread_user_pk': df2.get('thread_user_pk', df2.get('user_threads_userpk', df2['user_pk'])),
    'thread_follower_pk': df2.get('thread_follower_pk', df2['follower_pk']),
    'thread_username': df2.get('thread_username', pd.NA),
    'thread_follower_username': df2.get('thread_follower_username', df2['follower_username'])
})

# Concatenate
df = pd.concat([df1_mapped, df2_mapped], ignore_index=True)
df["thread_user_pk"] = df["thread_user_pk"].apply(lambda x: str(float(x)).split('.')[0] if pd.notna(x) else x)

df["thread_follower_pk"] = df["thread_follower_pk"].apply(lambda x: str(float(x)).split('.')[0] if pd.notna(x) else x)
# Show sample
print(df.head())

# Save to CSV
output_csv = 'extracted_four_tuples.csv'
df.to_csv(output_csv, index=False)
print(f'Saved extracted CSV to {output_csv}')


  thread_user_pk thread_follower_pk thread_username thread_follower_username
0    63310777769         1451797369        mlssfshn              stacidblack
1    63310777769         2384555054        mlssfshn                leotw4552
2    63310777769           17096723        mlssfshn                homeecmel
3    63310777769        55856570272        mlssfshn       handmade_market_ks
4    63310777769        33147008532        mlssfshn              l0dservices
Saved extracted CSV to extracted_four_tuples.csv


In [122]:
df

,thread_user_pk,thread_follower_pk,thread_username,thread_follower_username
0,63310777769,1451797369,mlssfshn,stacidblack
1,63310777769,2384555054,mlssfshn,leotw4552
2,63310777769,17096723,mlssfshn,homeecmel
3,63310777769,55856570272,mlssfshn,handmade_market_ks
4,63310777769,33147008532,mlssfshn,l0dservices
...,...,...,...,...
751459,66823423030,63468813061,elxargus_xyz,yan_03lipuas
751460,66823423030,63269391921,elxargus_xyz,lottie._.aashu
751461,66823423030,63269391921,elxargus_xyz,lottie._.aashu
751462,66823423030,63078210365,elxargus_xyz,daniza1996
